# Lecture 8: calibration on Irish data

**ECU44291 Quantitative Macroeconomics, Week 9.**

At the end of the last lecture we found $\beta$ by solving the model inside a root finder. Today is about the numbers that go in: where each one comes from, whether the model and the data measure the same thing, and how much the answers move when a choice changes. The data are the files you will use for M3 (the same as `data/` in your project). The sixty-cohort economy is on the slides; building it is the task for milestone 3.

This is the complete notebook, and I walk through most of it in lecture. 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# The data should sit next to this notebook in code/data/; on Colab they are read from GitHub.
DATA = "code/data/" if Path("code/data").exists() else \
    "https://raw.githubusercontent.com/joekopecky/ECU44291-lectures/main/week08/code/data/"

## Task 1: the numbers

Each file has a README entry (`code/data/README.md`): what it measures, the source and table code, and the caveats. Read it before you use a series.

**Intuition before running:** in 2025, what fraction of Irish GDP is GNI\*? More or less than three quarters?

In [ ]:
# your code here


Three measures of one ratio, and an $\alpha$ that depends on the years you average over. None of them is wrong: they measure different things. The slides go through what each one counts.

## Task 2: who is alive

The sixty-cohort model has households aged 21 to 80, working to 65 and retired from 66. The cohort sizes $\mu_j$ come from the population by age. The CSO's old-age dependency ratio is the population aged 65 and over divided by the population aged 15 to 64.

**Intuition before running:** Would we expect the model's ratio of retirees to workers (66 to 80 over 21 to 65) to be above or below the CSO's ratio?

In [ ]:
# your code here


Lower: the model leaves out everyone over 80 and counts 65-year-olds as workers. The two ratios are different objects, so compare the model with the model's own ratio, never with the CSO's.

## Task 3: the return your choices imply

The firm's first-order condition holds in every equilibrium of the model: $r + \delta = \alpha\, Y/K$. So once you have chosen $\alpha$, $\delta$ and a $K/Y$ target, you have also chosen the interest rate,
$$r = \frac{\alpha}{K/Y} - \delta,$$
before you solve anything. That makes it a check on whether your three choices fit together.

**Intuition before running:** with $\alpha = 0.69$ (2023) and $K/Y = 2.35$ (stock over GDP), is $r$ above or below 10% a year?

In [ ]:
# your code here


A real return of a few percent a year is plausible; 22% is not, and neither is 0.3%. A combination that implies a strange $r$ will need a strange $\beta$ to hit it.

## The two-period OLG economy, as in Lectures 2 and 7

The same functions as last week: log utility, $\beta = 0.9$, $\alpha = 0.3$, $\delta = 0$, $\tau_L = 0.2$, $\tau_K = 0.15$, no pension. A period is a generation, about 30 years.

In [ ]:
def params(**overrides):
    p = dict(alpha=0.3, A=1.0, beta=0.9, delta=0.0, tau_L=0.2, tau_K=0.15,
             t_y=0.0, t_o=0.0, tau_p=0.0, N_y=1.0, N_o=1.0)
    p.update(overrides)
    return p

def prices(K, p):
    L = p["N_y"]
    q = p["alpha"] * p["A"] * K ** (p["alpha"] - 1) * L ** (1 - p["alpha"])
    w = (1 - p["alpha"]) * p["A"] * K ** p["alpha"] * L ** (-p["alpha"])
    R = 1 + (1 - p["tau_K"]) * (q - p["delta"])
    return q, w, R

def pension_benefit(w, p):
    return p["tau_p"] * w * p["N_y"] / p["N_o"]

def savings(w, R, p):
    income_y = (1 - p["tau_L"] - p["tau_p"]) * w + p["t_y"]
    income_o = p["t_o"] + pension_benefit(w, p)
    return (p["beta"] * R * income_y - income_o) / ((1 + p["beta"]) * R)

def capital_supply(K, p):
    _, w, R = prices(K, p)
    return p["N_y"] * savings(w, R, p)

def solve_by_root(p):
    return brentq(lambda K: capital_supply(K, p) - K, 1e-3, 2.0, xtol=1e-12)

## Task 4: sensitivity

Move one parameter, hold the rest, solve the model again at each value, and plot what you care about. Here: $K/Y$ and $r$ against $\alpha$.

**Before you run:** as $\alpha$ rises, does $K/Y$ rise or fall? Why?

In [ ]:
# your code here


**The mechanism.** The young save out of wages, and the wage bill is a share $1 - \alpha$ of output. A higher $\alpha$ leaves them less to save from, so capital per unit of output falls: here $K/Y = \beta(1-\tau_L)(1-\alpha)/(1+\beta)$. With less capital, and a larger share of output paid to it, the return $r = \alpha\, Y/K$ rises.

"$K/Y$ falls as $\alpha$ rises" is what the plot shows. The paragraph above is why. A sensitivity plot needs the second.

**Before you run the next cell:** we change one word, `"alpha"` to `"tau_K"`, the tax on capital income. What will the two lines look like?

In [ ]:
# your code here


Flat. With log utility and no pension, saving is a fixed share of the wage (Lecture 7's saving rule, first term), and $R$ does not enter it. A tax on the return changes what the old consume, not what the young save. In your sixty-cohort household the same tax may matter: whether it does is a question about your model, and the plot answers it.

## Task 5 (homework): calibrate the two-period economy to Ireland

A period here is 30 years, so output per period is about 30 years of annual output, and an annual $K/Y$ of 3 is a $K/Y$ of 0.1 per period. For each of the three Irish $K/Y$ measures in Task 3, find the $\beta$ per period that hits it (with `brentq`, as in Lecture 7, Task 5), and the annual discount factor $\beta^{1/30}$ it corresponds to. Check one of them against the closed form $K/Y = \beta(1-\tau_L)(1-\alpha)/(1+\beta)$.

In [ ]:
# your code here


## Before the next lecture

- **M2, the proposal: Friday 13 November, 17:00.** Section 4, the calibration of each new parameter: its value or target, the source, and which parameter each target pins.
- **M3, the engine and its calibration: Tuesday 24 November, 17:00.** The data are in `data/` in your project, with the README.
- Task 5 above.
- Next lecture, Thursday 19 November: policy experiments and welfare by cohort.